# Đánh giá E5 + Reranker — corpus 5000 SP

Pipeline 2 giai đoạn:
1. **Bi-encoder** `e5_base_finetuned_5000` → retrieve top-**n**
2. **Cross-encoder reranker** → xếp hạng lại → lấy top-**k**

**Metrics:** Precision@k, Recall@k, **F1@k**, MRR@k, NDCG@k

> **Đánh giá trên `ecommerce.csv`:** query = `title`, corpus = `searchable_text`.  
> Script dò **Recall@n** trước → chọn `selected_n` đạt `--target-recall` → **chỉ rerank tại n đó** (không rerank max n).

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" "accelerate>=1.1.0" torch datasets pandas scikit-learn numpy tqdm einops

## 2) Setup: pull code + mount Drive + kiểm tra model

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
COLAB_REPO = Path("/content/llm_provider_benchmarking")

if COLAB_REPO.exists() and (COLAB_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO), "pull", "--ff-only"], check=False)
elif not (COLAB_REPO / "embedding_project" / "data" / "ecommerce.csv").is_file():
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)

REPO_DIR = COLAB_REPO
SCRIPTS = REPO_DIR / "embedding_project" / "scripts"
sys.path.insert(0, str(SCRIPTS))

# Model trên Drive (sửa path nếu khác)
from google.colab import drive
drive.mount("/content/drive")
EMB_MODEL = Path("/content/drive/MyDrive/models/e5_base_finetuned_5000")
RERANKER = Path("/content/drive/MyDrive/models/reranker")

EVAL_CSV = REPO_DIR / "embedding_project" / "data" / "ecommerce.csv"
OUTPUT_JSON = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_pipeline_eval.json"

for label, p in [("embedding", EMB_MODEL), ("reranker", RERANKER), ("csv", EVAL_CSV)]:
    print("OK" if p.exists() else "MISSING", label, "->", p)

## 3) Chọn n theo Recall, rồi mới rerank

1. Encode E5 → tính **Recall@n** trên các `n` trong `--n-search-values`
2. Chọn **n nhỏ nhất** đạt `--target-recall` (mặc định 0.98)
3. **Sau đó** mới load reranker → chỉ rerank tại `selected_n`
4. Grid k (5/10/20) từ cache — vài giây

Ví dụ: Recall@10 = 1.0 với 500 query → **500 × 10 = 5.000 cặp** (thay vì 50.000 @ n=100).

> `--skip-threshold`: bỏ FPR/FNR/EER. Pull script mới trước khi chạy.

In [ ]:
import torch

MAX_QUERIES = 500
QUERY_COL = "title"
N_VALUES = [10, 20, 30, 50, 75, 100]
N_SEARCH_VALUES = [10, 20, 30, 50, 75, 100]
TARGET_RECALL = 0.98
K_VALUES = [5, 10, 20]
EVAL_K = 10
RERANK_BATCH = 32

OUTPUT_JSON = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_pipeline_eval_dense_n.json"

cmd = [
    sys.executable,
    str(SCRIPTS / "evaluate_reranker_pipeline.py"),
    "--embedding-model", str(EMB_MODEL),
    "--reranker-model", str(RERANKER),
    "--eval-csv", str(EVAL_CSV),
    "--query-col", QUERY_COL,
    "--output", str(OUTPUT_JSON),
    "--eval-k", str(EVAL_K),
    "--target-recall", str(TARGET_RECALL),
    "--n-values", *[str(n) for n in N_VALUES],
    "--n-search-values", *[str(n) for n in N_SEARCH_VALUES],
    "--k-values", *[str(k) for k in K_VALUES],
    "--rerank-batch-size", str(RERANK_BATCH),
    "--skip-threshold",
]
if MAX_QUERIES:
    cmd.extend(["--max-queries", str(MAX_QUERIES)])

print("CUDA:", torch.cuda.is_available())
print(f"Recall@n → chọn n nhỏ nhất đạt {TARGET_RECALL}, rồi mới rerank tại n đó.")
print("Lệnh:", " ".join(cmd))
!{" ".join(cmd)}

## 4) Bảng kết quả & biểu đồ ngưỡng

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

EVAL_K = 10
RESULT_PATH = REPO_DIR / "embedding_project" / "outputs" / "evaluation" / "reranker_pipeline_eval_dense_n.json"

if not RESULT_PATH.is_file():
    raise FileNotFoundError(f"Chưa có kết quả: {RESULT_PATH}\nChạy cell 3 trước.")

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
best_metric = result.get("best_metric", "NDCG")
metric_key = f"F1@{EVAL_K}"

bi = result[f"bi_encoder_only@{EVAL_K}"]
rk_info = result[f"reranker_best_n@{EVAL_K}"]
rk = rk_info["metrics"]
opt = result["optimal_n_k"][f"by_max_{best_metric.lower()}_at_k{EVAL_K}"]
thr = result.get("threshold_analysis", {})
grid = pd.DataFrame(result["grid_search_n_k"])

def row_metrics(stage: str, m: dict) -> dict:
    return {
        "stage": stage,
        f"P@{EVAL_K}": m.get(f"Precision@{EVAL_K}", 0.0),
        f"R@{EVAL_K}": m.get(f"Recall@{EVAL_K}", 0.0),
        f"F1@{EVAL_K}": m.get(f"F1@{EVAL_K}", 0.0),
        f"MRR@{EVAL_K}": m.get(f"MRR@{EVAL_K}", 0.0),
        f"NDCG@{EVAL_K}": m.get(f"NDCG@{EVAL_K}", 0.0),
    }

summary = pd.DataFrame([
    row_metrics("Bi-encoder only", bi),
    row_metrics(f"Reranker (n={rk_info['n']})", rk),
])
print(f"=== So sánh @{EVAL_K} ({result['n_eval_queries']} queries, corpus {result['corpus_size']}) ===")
display(summary)

print(f"\n=== Top 5 (n, k) theo {metric_key} ===")
top = grid[grid["k"] == EVAL_K].sort_values(metric_key, ascending=False).head(5)
display(top)

opt_nk = result["optimal_n_k"]
print(f"=== selected_n = {result.get('selected_n')} (target recall) ===")
print("Recall@n:", json.dumps(opt_nk.get("recall_by_n", {}), indent=2))

print(f"\n=== Cấu hình tối ưu k (max {best_metric}@{EVAL_K}, n={result.get('selected_n')}) ===")
print(json.dumps(opt, ensure_ascii=False, indent=2))

if thr.get("EER"):
    print("\n=== Ngưỡng reranker ===")
    print("EER:", json.dumps(thr["EER"], indent=2))
    curve = pd.DataFrame(result.get("threshold_curve", []))
    if len(curve):
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(curve["threshold"], curve["FPR"], label="FPR")
        ax[0].plot(curve["threshold"], curve["FNR"], label="FNR")
        ax[0].legend()
        ax[0].set_title("FPR / FNR vs threshold")
        ax[1].plot(curve["threshold"], curve["error_rate"], color="crimson")
        ax[1].set_title("Error rate vs threshold")
        plt.tight_layout()
        plt.show()
else:
    print("\n(Bỏ qua ngưỡng — cell 3 dùng --skip-threshold)")

pivot = grid.pivot(index="n", columns="k", values=metric_key)
print(f"\n=== Bảng {metric_key} (n × k) ===")
display(pivot)

## 5) Giải thích cho báo cáo GVHD

- **n**: số ứng viên bi-encoder lấy ra (recall stage-1); chọn **n nhỏ nhất** đạt Recall ≥ target (0.98)
- **k**: số kết quả cuối sau rerank (grid 5/10/20)
- **F1@10** = 2·P·R/(P+R) — metric cân bằng hơn P@10 đơn lẻ khi 1 nhãn/query
- **EER**: ngưỡng τ sao cho FPR(τ) ≈ FNR(τ) — điểm cân bằng false alarm / miss
- **Min error**: τ làm (FP+FN) nhỏ nhất trên cặp (query, passage) có nhãn